# echoSentinel v2 — Train the PANNs+PCEN engine on a free GPU

Runs on **Google Colab** (Runtime -> Change runtime type -> **T4 GPU**) or Kaggle.

Steps: upload the repo + Dataset, install, fetch the pretrained CNN14 backbone once,
train, then download the trained weights back to your machine into `weights/`.

Checkpointing is per-epoch on best val F1, so a Colab disconnect never loses the best model.

## 1. Get the code and data onto the machine

**On your PC first**, build the lean upload bundle — it packs the code + training data but *excludes* the multi-GB official test sets you don't need for training:

```powershell
cd echosentinel_v2
python scripts/make_colab_bundle.py      # -> echosentinel_v2_colab.zip (~1-2 GB)
```

Upload `echosentinel_v2_colab.zip` to your Google Drive (MyDrive root), then run the cell below.

In [ ]:
from google.colab import drive
import os, glob
drive.mount('/content/drive')

MYDRIVE = '/content/drive/MyDrive'

def find_in_drive(name):
    """Locate a file by name under My Drive, whether it's a real file or a
    shortcut that resolves to a nested path."""
    direct = os.path.join(MYDRIVE, name)
    if os.path.exists(direct):
        return direct
    hits = glob.glob(f'{MYDRIVE}/**/{name}', recursive=True)
    return hits[0] if hits else None

# --- Full bundle (code + training data) ---
full = find_in_drive('echosentinel_v2_colab.zip')
assert full, ("echosentinel_v2_colab.zip not found under My Drive. Make sure the "
              "shortcut is added to *My Drive* (not just 'Shared with me').")
print('Data bundle:', full)
!cp "{full}" /content/
!cd /content && unzip -q -o echosentinel_v2_colab.zip

# --- Optional code refresh (tiny zip, contains the latest fixes) ---
code = find_in_drive('echosentinel_v2_code.zip')
if code:
    print('Code refresh:', code)
    !cp "{code}" /content/
    !cd /content && unzip -q -o echosentinel_v2_code.zip
    print('Applied code refresh from echosentinel_v2_code.zip')
else:
    print('No code refresh zip found — using code from the full bundle.')

%cd /content/echosentinel_v2
print('\nDataset folders:')
!ls Dataset

## 2. Install the package

In [ ]:
!pip install -q -e .
# Colab preinstalls a newer pandas that conflicts with our stack; pin it.
!pip install -q 'pandas>=2.0,<2.4'
import torch; print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Build manifests (if not already present) and fetch the pretrained backbone

The audit CSV ships in the repo; rebuild the manifest in case paths differ, then
download the CNN14 AudioSet checkpoint once (~310 MB).

In [ ]:
!python scripts/00_audit_dataset.py
!python scripts/01_build_manifests.py
!python scripts/fetch_pretrained.py

## 4. Train PANNs + PCEN

~60 epochs on a T4 is roughly 1–2 hours. Lower `--epochs` for a quick run.

**The checkpoint is written DIRECTLY to your Drive** on every val-F1 improvement
(`--weights-out`), so even if the session dies or the quota cuts you off mid-run,
the best model so far is already saved — no separate save step needed.

In [ ]:
!python scripts/04_train.py --config configs/model_panns.yaml \
    --weights-out '/content/drive/MyDrive/panns_pcen.pt'

## 5. Evaluate locally with the exact competition metric (IER)

Build a synthetic validation set that mirrors the noisy test conditions, run the
trained model, and score IER.

In [ ]:
!python scripts/03_build_synth_valset.py --n-scenes 24 --seconds 60
!python predict.py --input_dir out/synth_valset --output_json out/results.json --weights '/content/drive/MyDrive/panns_pcen.pt'
!python scripts/05_evaluate_ier.py --predictions out/results.json --ground-truth out/synth_valset/ground_truth.json

## 6. Done — weights are already on Drive

Training wrote the best checkpoint directly to `MyDrive/panns_pcen.pt` (step 4).
Just download it from Drive into your local repo's `weights/` folder.

Verify you have the right file locally before proceeding: its recorded best epoch /
val F1 should match what the training log printed at the end of step 4.